# Depth Estimation using Depth Anything


This notebook demonstrates the implementation and testing of a depth estimation model using a pre-trained deep learning model.


## Objective

The goal of this task is to estimate depth information from a monocular video.

Depth estimation allows the system to understand how far objects are from the camera, which is useful for scene understanding and decision-making.


## Model Description

We use a pre-trained model called **Depth Anything** for monocular depth estimation.

The model has been trained on large-scale datasets and can predict depth from a single image without requiring stereo input.

This notebook focuses on implementation, inference, and testing.


## Setup

Import required libraries.


In [30]:
!pip install opencv-python torch torchvision transformers matplotlib


In [31]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor, AutoModelForDepthEstimation


## Load Pre-trained Model

Load the model and move it to GPU if available.


In [32]:
processor = AutoImageProcessor.from_pretrained("LiheYoung/depth-anything-small-hf")
model = AutoModelForDepthEstimation.from_pretrained("LiheYoung/depth-anything-small-hf")

device = torch.device("cpu")
model.to(device)
model.eval()


Loading weights: 100%|██████████| 287/287 [00:00<00:00, 2991.06it/s]


DepthAnythingForDepthEstimation(
  (backbone): Dinov2Backbone(
    (embeddings): Dinov2Embeddings(
      (patch_embeddings): Dinov2PatchEmbeddings(
        (projection): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): Dinov2Encoder(
      (layer): ModuleList(
        (0-11): 12 x Dinov2Layer(
          (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
          (attention): Dinov2Attention(
            (attention): Dinov2SelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
            )
            (output): Dinov2SelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (layer_scale1): Di

## Depth Prediction Function

This function takes a frame and returns a normalized depth map.


In [33]:
def predict_depth(frame):

    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        predicted_depth = outputs.predicted_depth

    depth = predicted_depth.squeeze().cpu().numpy()
    depth = cv2.resize(depth, (frame.shape[1], frame.shape[0]))

    depth_norm = cv2.normalize(depth, None, 0, 255, cv2.NORM_MINMAX)
    depth_norm = depth_norm.astype(np.uint8)

    return depth_norm


## Video Processing

Process the video frame by frame and save depth output.


In [34]:
def process_video(input_path, output_path):

    cap = cv2.VideoCapture(input_path)

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height), True)


    while True:
        ret, frame = cap.read()
        if not ret:
            break

        depth_map = predict_depth(frame)

        #convert to color
        colored = cv2.applyColorMap(depth_map, cv2.COLORMAP_INFERNO)

        #save colored frame
        out.write(colored)


    cap.release()
    out.release()


## Testing

Run the model on a sample video.


In [35]:
input_video = "IMG_2561.mp4"
output_video = "output_depth.mp4"

process_video(input_video, output_video)


## Display Output Video


In [36]:
from IPython.display import Video
Video("output_depth.mp4", embed=True)


## Depth Distribution Visualization


In [37]:
def plot_depth_histogram(depth_map):
    plt.figure()
    plt.hist(depth_map.flatten(), bins=50)
    plt.title("Depth Value Distribution")
    plt.xlabel("Depth Value")
    plt.ylabel("Frequency")
    plt.show()

cap = cv2.VideoCapture("input.mp4")
ret, frame = cap.read()

if ret:
    depth_map = predict_depth(frame)
    plot_depth_histogram(depth_map)

cap.release()


## Original vs Depth Map


In [38]:
cap = cv2.VideoCapture("input.mp4")
ret, frame = cap.read()

if ret:
    depth_map = predict_depth(frame)
    colored = cv2.applyColorMap(depth_map, cv2.COLORMAP_INFERNO)

    combined = np.hstack((frame, colored))

    plt.figure(figsize=(10,5))
    plt.imshow(cv2.cvtColor(combined, cv2.COLOR_BGR2RGB))
    plt.title("Original vs Depth Map")
    plt.axis('off')
    plt.show()

cap.release()


## Real-Time Depth Estimation

Press ESC to exit.


In [39]:
def run_realtime_depth():

    cap = cv2.VideoCapture(0)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        depth_map = predict_depth(frame)

        colored = cv2.applyColorMap(depth_map, cv2.COLORMAP_INFERNO)
        combined = np.hstack((frame, colored))

        cv2.imshow("Real-Time Depth", combined)

        if cv2.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()

run_realtime_depth()
